# NLP Support Ticket Classifier — Model Development

This notebook documents the model-development workflow for a 27-class customer-support intent classifier.

**Goal:** predict `intent` from the customer's `instruction`.

Models compared:

1. TF-IDF + Logistic Regression
2. TF-IDF + Linear SVM
3. Calibrated Linear SVM
4. `all-MiniLM-L6-v2` sentence embeddings + Logistic Regression

The deployed application uses the semantic embedding model. See the repository README for the model-selection rationale and limitations.

## 1. Load the dataset

If the CSV is missing, run `python scripts/download_data.py` from the repository root first.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/customer_support.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing values:")
print(df.isna().sum())

## 2. Class distribution

In [ ]:
print("Number of intents:", df["intent"].nunique())
print("Number of categories:", df["category"].nunique())

df["intent"].value_counts().sort_index()

The intent classes are close to balanced, so **Macro F1** is a useful headline metric alongside accuracy.

## 3. Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df["instruction"]
y = df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## 4. Baseline — TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

pipeline_logistic = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)),
    ("model", LogisticRegression(max_iter=2000)),
])

pipeline_logistic.fit(X_train, y_train)
pred_logistic = pipeline_logistic.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_logistic))
print("Macro F1:", f1_score(y_test, pred_logistic, average="macro"))

Observed test result during development:

- Accuracy: **99.40%**
- Macro F1: **99.41%**

## 5. TF-IDF + Linear SVM

In [ ]:
from sklearn.svm import LinearSVC

pipeline_svm = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)),
    ("model", LinearSVC()),
])

pipeline_svm.fit(X_train, y_train)
pred_svm = pipeline_svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_svm))
print("Macro F1:", f1_score(y_test, pred_svm, average="macro"))

Observed test result during development:

- Accuracy: **99.52%**
- Macro F1: **99.52%**

## 6. Cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_logistic = cross_val_score(
    pipeline_logistic, X, y, cv=5, scoring="f1_macro"
)

cv_svm = cross_val_score(
    pipeline_svm, X, y, cv=5, scoring="f1_macro"
)

print("Logistic CV:", cv_logistic)
print("Logistic mean:", cv_logistic.mean())
print("Logistic std:", cv_logistic.std())

print("\nSVM CV:", cv_svm)
print("SVM mean:", cv_svm.mean())
print("SVM std:", cv_svm.std())

Observed 5-fold Macro F1:

| Model | Mean | Std |
|---|---:|---:|
| TF-IDF + Logistic Regression | 0.9925 | 0.00047 |
| TF-IDF + Linear SVM | 0.9943 | 0.00104 |

## 7. Calibrated Linear SVM

`LinearSVC` has no native `predict_proba`. Calibration was tested to obtain probability estimates.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

pipeline_svm_calibrated = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)),
    ("model", CalibratedClassifierCV(LinearSVC(), method="sigmoid", cv=5)),
])

pipeline_svm_calibrated.fit(X_train, y_train)
pred_calibrated = pipeline_svm_calibrated.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_calibrated))
print("Macro F1:", f1_score(y_test, pred_calibrated, average="macro"))

Observed test result:

- Accuracy: **99.48%**
- Macro F1: **99.48%**

## 8. Semantic sentence embeddings

The final approach uses `sentence-transformers/all-MiniLM-L6-v2`, which maps each message to a 384-dimensional semantic embedding.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

X_embeddings = embedding_model.encode(
    df["instruction"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
)

X_embeddings.shape

In [ ]:
X_train_emb, X_test_emb, y_train_emb, y_test_emb = train_test_split(
    X_embeddings,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

semantic_classifier = LogisticRegression(max_iter=3000)
semantic_classifier.fit(X_train_emb, y_train_emb)

pred_semantic = semantic_classifier.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_test_emb, pred_semantic))
print("Macro F1:", f1_score(y_test_emb, pred_semantic, average="macro"))

Observed test result:

- Accuracy: **99.40%**
- Macro F1: **99.40%**

The TF-IDF SVM was slightly stronger on the held-out dataset. The semantic model was selected for deployment after manual paraphrase tests exposed some TF-IDF failures.

This manual test is a useful diagnostic, but it is **not** an independent benchmark.

## 9. Manual robustness checks

In [ ]:
intent_to_category = (
    df[["intent", "category"]]
    .drop_duplicates()
    .set_index("intent")["category"]
    .to_dict()
)

def predict_semantic(text):
    embedding = embedding_model.encode([text], convert_to_numpy=True)
    probabilities = semantic_classifier.predict_proba(embedding)[0]
    index = probabilities.argmax()
    intent = semantic_classifier.classes_[index]

    return {
        "intent": intent,
        "category": intent_to_category[intent],
        "confidence": round(float(probabilities[index]) * 100, 1),
    }

tests = [
    "My order has not arrived yet",
    "I need help",
    "I want to cancel everything",
    "Where is my package?",
    "I forgot my password",
    "Can you change the address for my order?",
]

for text in tests:
    print(text)
    print(predict_semantic(text))
    print()

## 10. Train and save the final deployed model

After evaluation, the semantic Logistic Regression classifier is refit on all 26,872 messages. The sentence-transformer itself is loaded by model name in the Streamlit app.

In [ ]:
import joblib

final_semantic_classifier = LogisticRegression(max_iter=3000)
final_semantic_classifier.fit(X_embeddings, y)

joblib.dump(
    final_semantic_classifier,
    "../models/semantic_intent_classifier.joblib",
)

joblib.dump(
    intent_to_category,
    "../models/intent_to_category.joblib",
)

print("Saved final classifier and intent/category mapping.")

## Summary

Development results:

| Approach | Test Macro F1 |
|---|---:|
| TF-IDF + Logistic Regression | 99.41% |
| TF-IDF + Linear SVM | **99.52%** |
| Calibrated Linear SVM | 99.48% |
| MiniLM embeddings + Logistic Regression | 99.40% |

The final app uses the MiniLM semantic representation because of its behaviour on selected paraphrased examples, while the README explicitly documents that the SVM produced the best conventional held-out metric.

A stronger next evaluation step would be a separately curated external test set containing naturally written customer messages.